# Live demo — Outcome 2: what actually predicts fleet SoH

**Run this cell-by-cell in front of the panel.** It downloads the *exact,
unmodified* analysis script and its input data from this dissertation's own
public code repository and re-runs the model + SHAP explanation from scratch.

Repository: https://github.com/Usharani1699/battery-degradation-analytics

This reproduces the two charts used on the *"Outcome 2"* and *"Confirming
Outcome 2"* slides: mileage and duty-cycle intensity dominate fleet SoH;
FSI's composite contribution is real but modest — confirmed independently by
both a Random Forest permutation test (in the dissertation) and XGBoost + SHAP
(here).

In [ ]:
# 1) Install the packages the script needs
!pip -q install xgboost shap pandas numpy scikit-learn matplotlib

In [ ]:
# 2) Pull the real script + its input data straight from the public repo
import os, urllib.request

BASE = "https://raw.githubusercontent.com/Usharani1699/battery-degradation-analytics/main"
os.makedirs("03_Processed_Data", exist_ok=True)

files = {
    "03_Processed_Data/evbattery_vehicle_fsi.csv": f"{BASE}/08_Live_Demo_Notebooks/evbattery_vehicle_fsi.csv",
}
for local, url in files.items():
    urllib.request.urlretrieve(url, local)
    print(f"downloaded: {local}  ({os.path.getsize(local):,} bytes)")

In [ ]:
# 3) Reproduce the model + SHAP explanation live (same steps as
#    04_Code/evbattery_ml_analysis.py in the repo, condensed to what this
#    demo needs)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_absolute_error
import xgboost as xgb, shap

df = pd.read_csv("03_Processed_Data/evbattery_vehicle_fsi.csv")
print(f"Loaded {len(df)} real vehicles\n")

FEATURES = {
    'mileage_km'   : 'Mileage (km)',
    'ki_mean'      : 'KI (current variability)',
    't_avg_c'      : 'T_avg (\u00b0C)',
    'c_rate_mean'  : 'C-rate',
    'snippet_count': 'Snippet count',
    'FSI'          : 'FSI (composite)',
}
X = df[list(FEATURES.keys())].values
y = df['soh_pct'].values
feat_names = list(FEATURES.values())

model = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=42, verbosity=0)

# Honest, cross-validated performance (NOT a full-data fit)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2  = cross_val_score(model, X, y, cv=kf, scoring='r2')
cv_mae = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
print("=== XGBoost 5-fold CV (honest, held-out) ===")
print(f"  R²  = {cv_r2.mean():.3f} \u00b1 {cv_r2.std():.3f}")
print(f"  MAE = {(-cv_mae).mean():.3f} \u00b1 {(-cv_mae).std():.3f} %SoH")
print("  (This is the number to quote — not a full-data R², which is always")
print("   near-perfect for a flexible model and proves nothing about generalisation.)")

model.fit(X, y)
explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(X)

In [ ]:
# 4) The SHAP beeswarm — live, matching the presentation slide
shap.summary_plot(shap_vals, X, feature_names=feat_names, plot_type='dot', max_display=6)

### Reading this live

- The 5-fold CV R² printed above is the **honest** number — always quote this
  one, not a same-data fit.
- The beeswarm ranks the same way as the Random Forest permutation-importance
  chart in the dissertation: **mileage** and **duty-cycle intensity** dominate;
  **FSI's composite contribution is real but modest** — two independent
  methods, one conclusion.